In [ ]:
import numpy as np

# RK integrátor a derivace

In [ ]:
RK_COEFFICIENT_TABLES = {
    1: ([0], [[0]], [1]),
    2: ([0, 1 / 2], [[0, 0], [1 / 2, 0]], [0, 1]),
    3: ([0, 1 / 2, 1], [[0, 0, 0], [1 / 2, 0, 0], [-1, 2, 0]], [1 / 6, 2 / 3, 1 / 6]),
    4: (
        [0, 1 / 2, 1 / 2, 1],
        [[0, 0, 0, 0], [1 / 2, 0, 0, 0], [0, 1 / 2, 0, 0], [0, 0, 1, 0]],
        [1 / 6, 1 / 3, 1 / 3, 1 / 6],
    ),
    5: (
        [0, 1 / 5, 3 / 10, 4 / 5, 8 / 9, 1, 1],
        [
            [0, 0, 0, 0, 0, 0, 0],
            [1 / 5, 0, 0, 0, 0, 0, 0],
            [3 / 40, 9 / 40, 0, 0, 0, 0, 0],
            [44 / 45, -56 / 15, 32 / 9, 0, 0, 0, 0],
            [19372 / 6561, -25360 / 2187, 64448 / 6561, -212 / 729, 0, 0, 0],
            [9017 / 3168, -355 / 33, 46732 / 5247, 49 / 176, -5103 / 18656, 0, 0],
            [35 / 384, 0, 500 / 1113, 125 / 192, -2187 / 6784, 11 / 84, 0],
        ],
        [35 / 384, 0, 500 / 1113, 125 / 192, -2187 / 6784, 11 / 84, 0],
    ),
    6: (
        [0, 1 / 3, 2 / 3, 1 / 3, 1 / 2, 1 / 2, 1],
        [
            [0, 0, 0, 0, 0, 0, 0],
            [1 / 3, 0, 0, 0, 0, 0, 0],
            [0, 2 / 3, 0, 0, 0, 0, 0],
            [1 / 12, 1 / 3, -1 / 12, 0, 0, 0, 0],
            [-1 / 16, 9 / 8, -3 / 16, -3 / 8, 0, 0, 0],
            [0, 9 / 8, -3 / 8, -3 / 4, 1 / 2, 0, 0],
            [9 / 44, -9 / 11, 63 / 44, 18 / 11, 0, -16 / 11, 0],
        ],
        [11 / 120, 0, 27 / 40, 27 / 40, -4 / 15, -4 / 15, 11 / 120],
    ),
    # Chat GPT hallucinated these coefficients :D It does wild things
    "5GPT": (
        [0, 1 / 4, 3 / 8, 12 / 13, 1, 1 / 2],
        [
            [0, 0, 0, 0, 0, 0],
            [1 / 4, 0, 0, 0, 0, 0],
            [3 / 32, 9 / 32, 0, 0, 0, 0],
            [1932 / 2197, -7200 / 2197, 7296 / 2197, 0, 0, 0],
            [439 / 216, -8, 3680 / 513, -845 / 4104, 0, 0],
            [-8 / 27, 2, -3544 / 2565, 1859 / 4104, -11 / 40, 0],
        ],
        [16 / 135, 0, 6656 / 12825, 28561 / 56430, -9 / 50, 2 / 55],
    ),
    "6GPT": (
        [0, 1 / 4, 1 / 4, 1 / 2, 3 / 4, 1, 1],
        [
            [0, 0, 0, 0, 0, 0, 0],
            [1 / 4, 0, 0, 0, 0, 0, 0],
            [1 / 8, 1 / 8, 0, 0, 0, 0, 0],
            [0, 0, 1 / 2, 0, 0, 0, 0],
            [3 / 16, -3 / 8, 3 / 8, 9 / 16, 0, 0, 0],
            [-3 / 7, 8 / 7, 6 / 7, -12 / 7, 8 / 7, 0, 0],
            [7 / 90, 0, 32 / 90, 12 / 90, 32 / 90, 7 / 90, 0],
        ],
        [7 / 90, 0, 32 / 90, 12 / 90, 32 / 90, 7 / 90, 0],
    ),
}


In [ ]:
class RKp:
    def __init__(self, order: int = 4):
        """
        Args:
            t0 (float): Starting time
            h (float): Time step
            dydt (function): function in shape  dy/dt = f(t, y)
            tmax (float, optional): Stopping time of integration. Defaults to None.
            order (int, optional): Order of RK method used. Currently implemented are {1,2,3,4,5}. Defaults to 4.
        """

        self.order = order

        # load RK coefficients
        (c, A, b) = RK_COEFFICIENT_TABLES[order]
        (self.c, self.A, self.b) = (np.array(c), np.array(A), np.array(b))

    # initialization of values before integration process begins
    def Initialize(self, y0: np.ndarray, dydt):
        if len(y0) % 2 == 1:
            raise Exception("Wrong length of initial vector")
        self.y = y0.copy()
        self.ICS = y0.copy()
        self.y_history = [y0.copy()]
        self.dydt = dydt

    # integrate the given system
    def Integrate(self, t0: float, h: float, tmax: float = None):
        self.t0 = t0
        self.t = t0
        self.h = h

        while True:
            if self.t + h < tmax:
                self.NextStep()
            else:
                if tmax - self.t > 1e-14:
                    self.h = tmax - self.t
                    self.NextStep()
                break

    # calculate ks for general Butcher table
    def _getKs(self):
        k = np.zeros((len(self.c),) + np.shape(self.y))
        for i in range(len(self.c)):
            t = self.t + self.h * self.c[i]

            dy = np.zeros_like(self.y)
            for j in range(i):
                dy += self.A[i, j] * k[j]
            y = self.y + self.h * dy

            k[i] = self.dydt(t, y)
        return k.copy()

    # perform one step of RKp method
    def NextStep(self):
        # obtain k values
        k = self._getKs()
        # update y and t values
        self.y += self.h * sum(self.b[i] * k[i] for i in range(len(self.c)))
        self.t += self.h
        # save y value
        self.y_history.append(self.y.copy())
        return (self.t - self.t0) // self.h

    def GetHistory(self):
        return self.y_history

    @classmethod
    def GetAllImplemented(cls) -> dict:
        return RK_COEFFICIENT_TABLES.keys()


In [ ]:
def deriv_r_o4(arr, dx):
    res = np.zeros_like(arr)
    n = len(arr)

    if n < 5:
        raise ValueError("Pole musí mít alespoň 5 bodů pro 4. řád přesnosti.")

    # Central difference
    res[2:-2] = (-arr[4:] + 8 * arr[3:-1] - 8 * arr[1:-3] + arr[:-4]) / (12 * dx)

    # Borders with 4th order treatment
    res[0] = (-25*arr[0] + 48*arr[1] - 36*arr[2] + 16*arr[3] - 3*arr[4]) / (12 * dx)
    res[1] = (-3*arr[0] - 10*arr[1] + 18*arr[2] - 6*arr[3] + arr[4]) / (12 * dx)

    res[-1] = (25*arr[-1] - 48*arr[-2] + 36*arr[-3] - 16*arr[-4] + 3*arr[-5]) / (12 * dx)
    res[-2] = (3*arr[-1] + 10*arr[-2] - 18*arr[-3] + 6*arr[-4] - arr[-5]) / (12 * dx)

    return res

# Linearizovaná vlnová rovnice a její integrace
Integrujeme tento systém:

$$
    \partial_T \psi = -\pi,
$$
$$  
  \partial_T \phi_R = -\partial_R \pi + \gamma_2 (\partial_R\psi - \phi_R)
$$
$$
    \partial_T \pi = -\partial_R\phi_R - \frac{n-1}{R}\phi_R  + F\psi .
$$

Zatím uvažujeme $F = 0$.

## 1+1D

In [ ]:
# --- Parametry gridu ---
dr = 0.1
r_min, r_max = 0, 10.0
r = np.arange(r_min, r_max + dr, dr)
N = len(r)

# --- Parametry pulsu ---
r0 = 4.0  # Střed pulsu
sigma = 0.5  # Šířka pulsu

# --- Parametry integrace ---
tmin = 0.0
timestep = 0.005
tmax = 30.0


In [ ]:
def system_rhs_1d(r, dx, gamma2):
  def _system(t, y):
      """
      d psi /dt = - pi
      d phi /dt = - d pi/dr + gamma2 (d psi/dr - phi)
      d pi /dt = - d phi/dr
      + Sommerfeldova odchozí podmínka na pravém okraji
      """
      # 1. Rozbalení vektoru y na jednotlivá pole
      psi = y[0:N]
      phi = y[N : 2 * N]
      pi = y[2 * N : 3 * N]

      # 2. Inicializace derivací (rhs)
      dpsi_dt = np.zeros(N)
      dphi_dt = np.zeros(N)
      dpi_dt = np.zeros(N)

      # --- d psi /dt = -pi ---
      dpsi_dt = - pi


      # --- d phi /dt = - d pi/dr + gamma2 (d psi/dr - phi)  - ---
      dphi_dt[1:] = - deriv_r_o4(pi, dx)[1:] + gamma2 * (deriv_r_o4(psi, dx) - phi)[1:]
      # Sudost Psi v r zaručuje v počátku Phi (= dPsi/dr) = 0
      dphi_dt[0] = 0.0

      # --- d pi /dt = - d phi/dr - 2/R phi ---
      dpi_dt = - deriv_r_o4(phi, dx)
      

      # --- Sommerfeld (odchozí vlna) ---
      # d_T psi + d_R psi + psi/R = 0, tj. pi = phi + psi/R;
      # derivací podle času: d_T pi = d_T phi + (d_T psi)/R = d_T phi - pi/R
      last_dphi_dt = dphi_dt[-1]
      last_dpi_dt = dpi_dt[-1]
      

      dpi_dt[-1] = last_dphi_dt 
      dphi_dt[-1] = last_dpi_dt 


      # # --- Dirichlet ---
      # dpi_dt[-1] = 0.0
      # dphi_dt[-1] = 0.0

      # 3. Zabalení zpět do jednoho vektoru
      return np.concatenate([dpsi_dt, dphi_dt, dpi_dt])
  return _system



In [ ]:
# --- Výpočet počátečních polí ---
# f = exp(-(r-r0)^2 / sigma^2)
psi = np.exp(-np.power(r - r0, 2) / np.power(sigma, 2))

# pi inicializujeme jako derivaci f
phi = deriv_r_o4(psi, dr)

# xi inicializujeme jako 0 -- odpovídá statickému startu
pi = np.zeros(N)

## Můžeme mít i odcházející vlnu
# pi = -deriv_r_o4(f, dr)

# --- Zabalení pro třídu RKp ---
y0 = np.concatenate([psi, phi, pi])

if len(y0) % 2 != 0:
    # Pokud je délka lichá, upravíme grid o jeden bod, aby Initialize nehodilo chybu
    r = r[:-1]
    psi = psi[:-1]
    phi = phi[:-1]
    pi = pi[:-1]
    y0 = np.concatenate([psi, phi, pi])
    N = len(r)

print(f"Počáteční data připravena. Celková délka vektoru y0: {len(y0)} (3x {N})")

# Máme 4. řád derivace, proto raději 6. řád RK
solver = RKp(order=6)
solver.Initialize(y0, system_rhs_1d(r, dr, 0.1))
solver.Integrate(t0=tmin, h=timestep, tmax=tmax)

print(f"Integrace ukončena")

# Převedení historie na numpy array [časové_kroky, prostorové_body]
history_full = np.array(solver.GetHistory())

history = history_full[:, 0:N]  # extrahujeme pouze psi (prvních N prvků)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation


# --- ANIMACE ---
every_nth_frame = 50

fig, ax = plt.subplots(figsize=(10, 5))
(line,) = ax.plot(r, history[0, :], lw=2, color="firebrick")

ax.set_xlim(r_min, r_max)
ax.set_ylim(-1, 1)
ax.set_title("Animace výsledků")
ax.set_xlabel("Prostor (x)")
ax.set_ylabel("Amplituda (u)")
ax.grid(True, linestyle="--", alpha=0.6)


def update(frame):
    line.set_ydata(history[frame * every_nth_frame, :])
    return (line,)


ani = FuncAnimation(fig, update, frames=len(history) // every_nth_frame, interval=20*(every_nth_frame//10), blit=True, cache_frame_data=False)

plt.close()

from IPython.display import HTML
HTML(ani.to_jshtml())


## 3+1D

In [ ]:
# --- Parametry gridu ---
dr = 0.1
r_min, r_max = 0, 10.0
r = np.arange(r_min, r_max + dr, dr)
N = len(r)

# --- Parametry pulsu ---
r0 = 4.0  # Střed pulsu
sigma = 0.5  # Šířka pulsu

# --- Parametry integrace ---
tmin = 0.0
timestep = 0.005
tmax = 30.0


In [ ]:

def system_rhs_3d(r, dx, gamma2):
  def _system(t, y):
      """
      d psi /dt = - pi
      d phi /dt = - d pi/dr + gamma2 (d psi/dr - phi)
      d pi /dt = - d phi/dr - 2/R phi
      + Sommerfeldova odchozí podmínka na pravém okraji
      """
      # 1. Rozbalení vektoru y na jednotlivá pole
      psi = y[0:N]
      phi = y[N : 2 * N]
      pi = y[2 * N : 3 * N]

      # 2. Inicializace derivací (rhs)
      dpsi_dt = np.zeros(N)
      dphi_dt = np.zeros(N)
      dpi_dt = np.zeros(N)

      # --- d psi /dt = -pi ---
      dpsi_dt[:] = - pi


      # --- d phi /dt = - d pi/dr + gamma2 (d psi/dr - phi)  - ---
      dphi_dt[1:] = - deriv_r_o4(pi, dx)[1:] + gamma2 * (deriv_r_o4(psi, dx) - phi)[1:]
      # Sudost Psi v r zaručuje v počátku Phi (= dPsi/dr) = 0
      dphi_dt[0] = 0.0

      # --- d pi /dt = - d phi/dr - 2/R phi ---
      dphi_dr = deriv_r_o4(phi, dx)
      dpi_dt[1:] = - dphi_dr[1:] - 2 * phi[1:]/r[1:]
      # Pro Pi l'Hospitalem dostaneme podmínku v počátku
      dpi_dt[0] = - 3 * dphi_dr[0]


      # --- Sommerfeld (odchozí vlna) ---
      last_dphi_dt = dphi_dt[-1]
      last_dpi_dt = dpi_dt[-1]      

      dpi_dt[-1] = last_dphi_dt - pi[-1]/r[-1]
      dphi_dt[-1] = last_dpi_dt + pi[-1]/r[-1]


      # # --- Dirichlet ---
      # dpi_dt[-1] = 0.0
      # dphi_dt[-1] = 0.0

      # 3. Zabalení zpět do jednoho vektoru
      return np.concatenate([dpsi_dt, dphi_dt, dpi_dt])
  return _system



In [ ]:
# --- Výpočet počátečních polí ---
# f = exp(-(r-r0)^2 / sigma^2)
psi = np.exp(-np.power(r - r0, 2) / np.power(sigma, 2))

# pi inicializujeme jako derivaci f
phi = deriv_r_o4(psi, dr)

# xi inicializujeme jako 0 -- odpovídá statickému startu
pi = np.zeros(N)

## Můžeme mít i odcházející vlnu
# pi = -deriv_r_o4(f, dr)

# --- Zabalení pro třídu RKp ---
y0 = np.concatenate([psi, phi, pi])

if len(y0) % 2 != 0:
    # Pokud je délka lichá, upravíme grid o jeden bod, aby Initialize nehodilo chybu
    r = r[:-1]
    psi = psi[:-1]
    phi = phi[:-1]
    pi = pi[:-1]
    y0 = np.concatenate([psi, phi, pi])
    N = len(r)

print(f"Počáteční data připravena. Celková délka vektoru y0: {len(y0)} (3x {N})")

# Máme 4. řád derivace, proto raději 6. řád RK
solver = RKp(order=6)
solver.Initialize(y0, system_rhs_3d(r, dr, 0.1))
solver.Integrate(t0=tmin, h=timestep, tmax=tmax)

print(f"Integrace ukončena")

# Převedení historie na numpy array [časové_kroky, prostorové_body]
history_full = np.array(solver.GetHistory())

history = history_full[:, 0:N]  # extrahujeme pouze psi (prvních N prvků)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation


# --- ANIMACE ---
every_nth_frame = 50

fig, ax = plt.subplots(figsize=(10, 5))
(line,) = ax.plot(r, history[0, :], lw=2, color="firebrick")

ax.set_xlim(r_min, r_max)
ax.set_ylim(-1, 1)
ax.set_title("Animace výsledků")
ax.set_xlabel("Prostor (x)")
ax.set_ylabel("Amplituda (u)")
ax.grid(True, linestyle="--", alpha=0.6)


def update(frame):
    line.set_ydata(history[frame * every_nth_frame, :])
    return (line,)


ani = FuncAnimation(fig, update, frames=len(history) // every_nth_frame, interval=20*(every_nth_frame//10), blit=True, cache_frame_data=False)

plt.close()

from IPython.display import HTML
HTML(ani.to_jshtml())
